# MILESTONE 2

In [27]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, pipeline
)
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── LOAD DATA ──────────────────────────────────────────────────
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')['train']

# Q1 - combined_text at index 51
dataset = dataset.map(lambda row: {'combined_text': row['prompt'] + ' ' + row['A']})
q1 = len(dataset[51]['combined_text'])
print(f"Q1 - combined_text length at index 51: {q1}")

# ── BERT TOKENIZER ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Q2 - vocab size
q2 = tokenizer.vocab_size
print(f"Q2 - BERT vocab size: {q2}")

# Q3 - SEP token ID
q3 = tokenizer.sep_token_id
print(f"Q3 - [SEP] token ID: {q3}")

# # Q4 - shape of input_ids when tokenizing full prompt column
# prompts = dataset['prompt']
# encoded = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
# q4 = tuple(encoded['input_ids'].shape)
# print(f"Q4 - input_ids shape: {q4}")

prompts = list(dataset['prompt'])  # ← add list()
encoded = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
q4 = tuple(encoded['input_ids'].shape)
print(f"Q4 - input_ids shape: {q4}")

# ── BERT ARCHITECTURE ──────────────────────────────────────────

# Q5 - attention head dimensionality
q5 = 768 // 12
print(f"Q5 - each attention head dimension: {q5}")

# Q6 - last_hidden_state shape for row 0
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = dataset[0]['prompt']
inputs = tokenizer(row0_prompt, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

q6 = tuple(outputs.last_hidden_state.shape)
print(f"Q6 - last_hidden_state shape: {q6}")

# Q7 - sum of first 5 values of [CLS] embedding
cls_vector = outputs.last_hidden_state[0, 0, :]
q7 = round(cls_vector[:5].sum().item(), 4)
print(f"Q7 - sum of first 5 CLS values: {q7}")

# Q8 - attention weight [CLS] pays to 'fusion'
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

test_string = "Light-ion fusion is a technique."
inputs_attn = tokenizer(test_string, return_tensors='pt')

# find token index of 'fusion'
tokens = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
print(f"  tokens: {tokens}")
fusion_idx = tokens.index('fusion')
print(f"  fusion token index: {fusion_idx}")

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

# last layer (index -1), first head (index 0), CLS (row 0) -> fusion token
last_layer_attn = outputs_attn.attentions[-1]  # (1, 12, seq_len, seq_len)
q8 = round(last_layer_attn[0, 0, 0, fusion_idx].item(), 4)
print(f"Q8 - attention weight [CLS] -> fusion: {q8}")

# ── SENTENCE TRANSFORMERS ──────────────────────────────────────
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Q9 - cosine similarity between prompt and option B for row 0
prompt_emb = sbert.encode(dataset[0]['prompt'])
optB_emb   = sbert.encode(dataset[0]['B'])
q9 = round(util.cos_sim(prompt_emb, optB_emb).item(), 4)
print(f"Q9 - cosine similarity (prompt vs B, row 0): {q9}")

# Q10 - MAP@3 for MiniLM pipeline + count where MiniLM finds correct but TF-IDF doesn't

import pandas as pd
train_df = pd.DataFrame(dataset)

options = ['A', 'B', 'C', 'D', 'E']

def map_at_3(actuals, preds_top3):
    scores = []
    for actual, top3 in zip(actuals, preds_top3):
        score = 0.0
        for rank, p in enumerate(top3):
            if p == actual:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# TF-IDF pipeline
tfidf = TfidfVectorizer(max_features=5000)
train_df['full_text'] = train_df.apply(lambda r: r['prompt'] + ' ' + r['A'] + ' ' + r['B'] + ' ' + r['C'] + ' ' + r['D'] + ' ' + r['E'], axis=1)
X_all = tfidf.fit_transform(train_df['full_text'])

tfidf_top3 = []
for i, row in train_df.iterrows():
    prompt_vec = tfidf.transform([row['prompt']])
    sims = []
    for opt in options:
        opt_vec = tfidf.transform([row[opt]])
        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        sims.append(sim)
    top3_idx = np.argsort(sims)[::-1][:3]
    tfidf_top3.append([options[j] for j in top3_idx])

# MiniLM pipeline
print("Encoding all prompts and options with MiniLM (this takes a few minutes)...")
prompt_embs = sbert.encode(train_df['prompt'].tolist(), batch_size=64, show_progress_bar=True)
option_embs = {opt: sbert.encode(train_df[opt].tolist(), batch_size=64, show_progress_bar=False) for opt in options}

minilm_top3 = []
for i in range(len(train_df)):
    sims = [util.cos_sim(prompt_embs[i], option_embs[opt][i]).item() for opt in options]
    top3_idx = np.argsort(sims)[::-1][:3]
    minilm_top3.append([options[j] for j in top3_idx])

actuals = train_df['answer'].tolist()
q10_map3 = round(map_at_3(actuals, minilm_top3), 4)
print(f"Q10a - MiniLM MAP@3: {q10_map3}")

# count where TF-IDF misses but MiniLM hits
count = 0
for i, actual in enumerate(actuals):
    tfidf_hit   = actual in tfidf_top3[i]
    minilm_hit  = actual in minilm_top3[i]
    if not tfidf_hit and minilm_hit:
        count += 1
print(f"Q10b - TF-IDF misses but MiniLM hits: {count}")

# ── ZERO-SHOT CLASSIFICATION ───────────────────────────────────
zsc = pipeline('zero-shot-classification')  # defaults to facebook/bart-large-mnli

row1 = dataset[1]
candidate_labels = [row1['A'], row1['B'], row1['C']]
prompt_text = row1['prompt']

# Q11 - softmax (default)
result_softmax = zsc(prompt_text, candidate_labels=candidate_labels)
print(f"\n  Zero-shot softmax scores: {result_softmax['scores']}")
print(f"  Labels order: {result_softmax['labels']}")
q11 = round(result_softmax['scores'][0], 4)
print(f"Q11 - top-ranked option probability (softmax): {q11}")

# Q12 - multi_label sigmoid
result_sigmoid = zsc(prompt_text, candidate_labels=candidate_labels, multi_label=True)
print(f"  Zero-shot sigmoid scores: {result_sigmoid['scores']}")
sum_softmax  = sum(result_softmax['scores'])
sum_sigmoid  = sum(result_sigmoid['scores'])
q12 = round(abs(sum_softmax - sum_sigmoid), 4)
print(f"Q12 - absolute diff of sum of probs (softmax vs sigmoid): {q12}")

# ── GENERATIVE QA ──────────────────────────────────────────────
# from transformers import pipeline as hf_pipeline

# gen = hf_pipeline('text2text-generation', model='google/flan-t5-small')
# row0 = dataset[0]
# input_str = f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."
# result_gen = gen(input_str, max_new_tokens=5)
# q13 = result_gen[0]['generated_text']
# print(f"\nQ13 - flan-t5-small output: '{q13}'")

from transformers import T5ForConditionalGeneration, T5Tokenizer

model_name = 'google/flan-t5-small'
t5_tokenizer = T5Tokenizer.from_pretrained(model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(model_name)

row0 = dataset[0]
input_str = f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."

inputs = t5_tokenizer(input_str, return_tensors='pt')
with torch.no_grad():
    outputs = t5_model.generate(**inputs, max_new_tokens=5)

q13 = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Q13 - flan-t5-small output: '{q13}'")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 - combined_text length at index 51: 614
Q2 - BERT vocab size: 30522
Q3 - [SEP] token ID: 102
Q4 - input_ids shape: (2000, 128)
Q5 - each attention head dimension: 64


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 - flan-t5-small output: 'B'
